In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re

from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import confusion_matrix, classification_report, RocCurveDisplay, PrecisionRecallDisplay, balanced_accuracy_score, brier_score_loss, roc_auc_score, f1_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.svm import SVC
from datetime import date
from enum import Enum

In [ ]:
dataframe = pd.read_csv('OGSA_PROMs_2023_spine.csv')
dataframe

In [ ]:
def computeCronbachsAlpha(questionnaire: str) -> float:
    ansDataframe = dataframe.filter(regex=f"^{questionnaire}_.*_ans_0$")

    questionVariance = ansDataframe.filter(regex = "_ans_0$").var()
    totalVariance    = ansDataframe.sum(axis=1).var()
    nQuestions       = len(questionVariance)
    
    return (nQuestions / (nQuestions - 1)) * (1 - (questionVariance.sum() / totalVariance)) 

In [ ]:
alphaODI = computeCronbachsAlpha("ODI")
alphaCOMI = computeCronbachsAlpha("COMI")
alphaSF36 = computeCronbachsAlpha("SF36")

print(f"Alpha ODI: {alphaODI:.2f}")
print(f"Alpha COMI: {alphaCOMI:.2f}")
print(f"Alpha SF36: {alphaSF36:.2f}")

In [ ]:
dataframe.filter(regex = "[^_]ans[^_]") # not an answer, answers only with _ans_

In [ ]:
colsAns = dataframe.filter(regex = "_ans_").columns
colsAns

In [ ]:
dataframe = dataframe.drop(columns = colsAns)
dataframe # 505 columns removed

In [ ]:
dataframe = dataframe.drop(columns = ["insert_year", "hospitalization_year"])
dataframe 

In [ ]:
cols2 = dataframe.filter(regex = "_2$")
cols3 = dataframe.filter(regex = "_3$")

dataframe = dataframe.drop(columns = cols2)
dataframe = dataframe.drop(columns = cols3)

dataframe

In [ ]:
dataframe = dataframe.drop(columns = ["surgery_time_1", "suture_time_1"])  # I don't think they are useful
dataframe

Age distribution plot

In [ ]:
sns.histplot(
    data=dataframe, 
    x="age_at_enrollment", 
    bins=20, 
    kde=True, 
    color="#1e894b",            
    line_kws={'color': '#7b3294', 'linewidth': 3} 
)
plt.title("Age at Enrollment Distribution", fontsize = 10)
plt.xlabel("Age", fontsize = 8)
plt.ylabel("Counts", fontsize = 8)
plt.tight_layout()
plt.savefig("age_distribution.png", dpi = 300, bbox_inches='tight')
plt.show()

Disease distribution plot

In [ ]:
sns.countplot(
    data = dataframe, 
    y = "category", 
    order = dataframe["category"].value_counts().index, 
    palette = "viridis",
    hue = "category",
    legend = False
)

plt.title(f"Disease Distribution", fontsize = 10)
plt.xlabel("Count", fontsize = 8)
plt.ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
sns.countplot(
    data = dataframe, 
    y = "category", 
    order = dataframe["category"].value_counts().index, 
    palette = "viridis",
    hue = "category",
    legend = False
)

plt.xscale("log")

plt.title("Disease Distribution (Log Scale)", fontsize = 10)
plt.xlabel("Count", fontsize = 8)
plt.ylabel('')
plt.tight_layout()

plt.savefig("disease_distribution_log.png", dpi = 300, bbox_inches='tight') 
plt.show()

Distribution of International Classification of Diseases code (standardized alphanumeric code system maintained by the World Health Organization that classifies diseases, injuries, and health conditions)

In [ ]:
dataframe.filter(like = "ICD_").columns

In [ ]:
ICD_colsToPlot = ["ICD_1", "ICD_code_PREOP_1", "ICD_code_INTRAOP_1", "ICD_code_POSTOP_1"] 
plt.figure(figsize=(8, 5))
for i, col in enumerate(ICD_colsToPlot):
    if col in dataframe.columns:
        plt.subplot(2, 2, i + 1)
        sns.histplot(dataframe[col], kde=True, color="#27ae37")
        plt.title(f'Distribution of {col} (log)')
        plt.yscale("log")
plt.tight_layout()
plt.show()

In [ ]:
colsCorr = ['age_at_enrollment', 'category']
corrDataFrame = dataframe[colsCorr].copy()

corrDataFrame = pd.get_dummies(corrDataFrame, columns=["category"], drop_first=True)

corrDataFrame.columns = [col.replace('category_', '').replace('_', ' ').title() for col in corrDataFrame.columns]
corrMatrix = corrDataFrame.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(
    corrMatrix, 
    fmt=".2f", 
    cmap='PRGn', 
    linewidths=0.5, 
    center=0
)

plt.title("Correlation Analysis: Age vs. Disease Categories", fontsize=14, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0) 

plt.tight_layout()

plt.savefig("Age-Disease_Correlation.png", dpi=300, bbox_inches='tight')
plt.show()

negative correlation:
- age / idiopatic deformity
- age / lumbar herniation

In [ ]:
colsToPlot = dataframe.filter(regex = "_0no_1yes_?1").columns
colsToPlot

In [ ]:
# rename the column with the correct name
dataframe.rename(columns = {"implants_0no_1yes1": "implants_0no_1yes_1"}, inplace = True)

In [ ]:
tec_colsToPlot = dataframe.filter(regex = "_0no_1yes_1").columns
newDF = dataframe[tec_colsToPlot].copy()

# newDF[colsToPlot].isna().sum()
newDF.dropna(inplace = True)
newDF

In [ ]:
df = newDF.melt(var_name='variable', value_name='value')

# clean the variable names for better readability
df['variable'] = df['variable'].str.replace('_0no_1yes_1', '').str.replace('_', ' ').str.title()

colors = {0: "#27ae60", 1: "#7b3294"}

plt.figure(figsize=(12, 6))
sns.countplot(data=df, x='variable', hue='value', palette=colors)

plt.title('Distribution of Surgical and Diagnostic Features', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Medical Variable', fontsize=12)
plt.ylabel('Count (Log Scale)', fontsize=12)

plt.yscale('log')
plt.xticks(rotation=45, ha='right')

# Add legend with custom labels and move it outside the plot
plt.legend(title='Value', labels=['No (0)', 'Yes (1)'], bbox_to_anchor=(1, 1), loc='upper left')

plt.tight_layout() # adjust layout to prevent clipping of labels

plt.savefig('Distribution of Surgical and Diagnostic Features.png', dpi=300, bbox_inches='tight') # save the figure with high resolution
plt.show()


In [ ]:
tecCols = dataframe.filter(regex = "_0no_1yes_1").columns.tolist()
otherCols = ["age_at_enrollment", "category", "ICD_1", "sex_0M_1F", "BMI_Total_PreOp", "Blood_Loss", "Blood_Transfusion"]

corrDataFrame = dataframe[tecCols + otherCols].copy()

corrDataFrame.columns = [col.replace('_0no_1yes_1', '').replace('_', ' ').title() for col in corrDataFrame.columns]

corrDataFrame = pd.get_dummies(corrDataFrame, columns = ["Category"], drop_first = True)
corrMatrix = corrDataFrame.corr()

# Rename the index and columns to remove "Category " or "Category_"
corrMatrix.index = [col.replace('Category ', '').replace('Category_', '') for col in corrMatrix.index]
corrMatrix.columns = [col.replace('Category ', '').replace('Category_', '') for col in corrMatrix.columns]

plt.figure(figsize = (12, 10))
sns.heatmap(
    corrMatrix, 
    fmt = ".2f", 
    cmap = 'PRGn',    
    linewidths = 0.5,
    center = 0   
)

plt.title("Correlation Matrix: Techniques vs Demographics", fontsize = 14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
ODIcols = ["ODI_Total_PreOp", "ODI_Total_3months", "ODI_Total_6months", 
            "ODI_Total_12months", "ODI_Total_24months", "ODI_Total_60months"]

plt.figure(figsize = (12, 6))
sns.boxplot(data = dataframe[ODIcols], palette = "viridis")
plt.xticks(rotation = 45)
plt.title("ODI evolution over time")
plt.ylabel("ODI scores")
plt.grid(axis = 'y', linestyle = '--', alpha = 0.7)
plt.show()

count = dataframe[ODIcols].isnull().sum()
na_ODI = {col.split("_")[-1]: int(count[col]) for col in ODIcols}
print(na_ODI)

In [ ]:
plt.figure(figsize=(10, 6))
sns.pointplot(
    data=dataframe[ODIcols], 
    capsize=0.1,           
    errorbar=("ci", 95),   # Shows the 95% Confidence Interval
    markers="o",           
    linestyles="-", 
    color="#21918c"   
)

plt.title("ODI Score Evolution Over Time", fontsize=14, fontweight='bold')
plt.xlabel("Follow-up Interval", fontsize=12)
plt.ylabel("Mean ODI Score (95% CI)", fontsize=12)

plt.xticks(
    ticks=range(len(ODIcols)), 
    labels=[col.split("_")[-1] for col in ODIcols], 
    rotation=45
)

plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()

plt.show()

In [ ]:
colsPointPlot = ["ODI_Total_PreOp", "ODI_Total_3months", "ODI_Total_6months", 
            "ODI_Total_12months", "ODI_Total_24months", "ODI_Total_60months",
            "COMI_Total_PreOp", "COMI_Total_3months", "COMI_Total_6months", 
            "COMI_Total_12months", "COMI_Total_24months", "COMI_Total_60months",
            "SF36_GeneralHealth_PreOp", "SF36_GeneralHealth_3months", "SF36_GeneralHealth_6months", 
            "SF36_GeneralHealth_12months", "SF36_GeneralHealth_24months", "SF36_GeneralHealth_60months"]

point plot con intervalli di confidenza

Let's check if the completion dates of the questionnaires are correct (how many days between 3months and 6months?)

In [ ]:
dataframe["ODI_completion_date_3months"] = pd.to_datetime(dataframe["ODI_completion_date_3months"])
dataframe["ODI_completion_date_6months"] = pd.to_datetime(dataframe["ODI_completion_date_6months"])

negativeRows = dataframe[(dataframe["ODI_completion_date_6months"] - dataframe["ODI_completion_date_3months"]).dt.days < 0]

columnsODI = ["ODI_completion_date_3months", "ODI_completion_date_6months"]
negativeRows = negativeRows[columnsODI]

print(negativeRows)

In [ ]:
def computeDatesDiffs(questionnaire: str) -> pd.DataFrame:
    months = ("PreOp", 3, 6, 12, 24, 60)
    monthsCols = [
        f"{questionnaire}_completion_date_{m if m != 'PreOp' else 'PreOp'}" + 
        ("" if m == 'PreOp' else "months") 
        for m in months
    ]
    
    dataframeDatesDiff = pd.DataFrame()

    for pos in range(len(monthsCols) - 1):
        nextDate     = pd.to_datetime(dataframe[monthsCols[pos + 1]])
        previousDate = pd.to_datetime(dataframe[monthsCols[pos]])
        difference   = (nextDate - previousDate).dt.days

        colName = f"{questionnaire}_{months[pos]}-{months[pos+1]}"
        dataframeDatesDiff[colName] = difference

    return dataframeDatesDiff

In [ ]:
def removeOutliers(datesDiff: pd.DataFrame) -> pd.DataFrame:
    datesDiffClean = datesDiff.copy()
    for col in datesDiff.columns:
        diff = datesDiffClean[col]
        diff = diff[diff > 0]
        
        q1 = diff.quantile(0.25)
        q3 = diff.quantile(0.75)
        iqr = q3 - q1
        
        lowerBound = q1 - 1.5 * iqr
        upperBound = q3 + 1.5 * iqr
        datesDiffClean[col] = diff[diff.between(lowerBound, upperBound)]

    return datesDiffClean

In [ ]:
datesDiff = computeDatesDiffs("ODI")
print(removeOutliers(datesDiff))

In [ ]:
def createViolinPlots(cleanData: pd.DataFrame) -> None:
    for col in cleanData.columns:
        name     = col.split("_")[0]   
        interval = col.split("_")[1]
        
        dates        = [d.strip() for d in interval.split("-")]
        date1, date2 = dates[0], dates[1]
        dataSeries   = cleanData[col]

        mean      = dataSeries.mean()
        median    = dataSeries.median()
        nPatients = dataSeries.count()

        fig, ax = plt.subplots(figsize=(10, 6))
        sns.violinplot(data = cleanData[col], ax = ax, palette = "viridis")

        plt.subplots_adjust(right=0.75)  # adding space on the right for the text
        textstr = ("STATISTICS\n"
                    f"Mean: {mean:.2f}\n"
                    f"Median: {median:.2f}\n"
                    f"Number of patients: {nPatients}")

        # add the text box with the statistics
        ax.text(1.05, 0.5, textstr, transform=ax.transAxes, fontsize=8, verticalalignment='center', 
                bbox=dict(boxstyle='round', 
                          facecolor='none', 
                          edgecolor='gray', 
                          alpha=0.5))

        plt.title(f"{name} - Days between answers at {date1} and {date2} months")
        plt.ylabel("days")
        plt.grid(axis = 'y', linestyle = '--', alpha = 0.7)
        plt.show()

FABQ has only PreOp values

In [ ]:
datesDiff_ODI = computeDatesDiffs("ODI")
cleanData_ODI = removeOutliers(datesDiff_ODI)

datesDiff_COMI = computeDatesDiffs("COMI")
cleanData_COMI = removeOutliers(datesDiff_COMI)

datesDiff_SF36 = computeDatesDiffs("SF36")
cleanData_SF36 = removeOutliers(datesDiff_SF36)

datesDiff_VAS = computeDatesDiffs("VAS")
cleanData_VAS = removeOutliers(datesDiff_VAS)

datesDiff_SRS22 = computeDatesDiffs("SRS22")
cleanData_SRS22 = removeOutliers(datesDiff_SRS22)

merging everything in a dataframe, only considering the 3-6months range

In [ ]:
filteredData = {
    "ODI"  : cleanData_ODI.filter(like="_3-6").iloc[:, 0],
    "COMI" : cleanData_COMI.filter(like="_3-6").iloc[:, 0],
    "SF36" : cleanData_SF36.filter(like="_3-6").iloc[:, 0],
    "VAS"  : cleanData_VAS.filter(like="_3-6").iloc[:, 0],
    "SRS22": cleanData_SRS22.filter(like="_3-6").iloc[:, 0]
}

mergedDF_36 = pd.concat(filteredData, axis=1)
mergedDF_36

In [ ]:
def createComparisonViolinPlots(mergedDF: pd.DataFrame, intervalName: str) -> None:
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.violinplot(data = mergedDF, ax = ax, palette = "viridis")

    textstr = f"STATISTICS ({intervalName})\n" # for adding the statistics of all the quest. in the same plot

    for col in mergedDF.columns:
        mean      = mergedDF[col].mean()
        median    = mergedDF[col].median()
        nPatients = mergedDF[col].count()

        textstr += (f"\n--- {col} ---\n"
                    f"Mean: {mean:.2f}\n"
                    f"Median: {median:.2f}\n"
                    f"N patients: {nPatients}\n")

    plt.subplots_adjust(right=0.75)  # adding space on the right for the text

    # add the text box with the statistics
    ax.text(1.05, 0.5, textstr, transform=ax.transAxes, fontsize=8, verticalalignment='center', 
            bbox=dict(boxstyle='round', 
                        facecolor='none', 
                        edgecolor='gray', 
                        alpha=0.5))

    plt.title(f"Comparison of Questionnaires: {intervalName} Months Follow-up")
    plt.ylabel("days")
    plt.grid(axis = 'y', linestyle = '--', alpha = 0.7)
    plt.savefig(f"Comparison_Questionnaires_{intervalName}_Months.png", dpi = 300, bbox_inches='tight')
    plt.show()

In [ ]:
createComparisonViolinPlots(mergedDF_36, "3 - 6")

lot of outliers, maybe I can remove just the wrong dates (negative ones? there were 4 for ODI, I can check the others)
ODI, COMI, SF36 good distribution (less than 80 days? low, but okay) -> 90 days expected with a 3 months gap

Now I do the same for PreOp-3months

In [ ]:
filteredData2 = {
    "ODI"  : cleanData_ODI.filter(like="_PreOp-3").iloc[:, 0],
    "COMI" : cleanData_COMI.filter(like="_PreOp-3").iloc[:, 0],
    "SF36" : cleanData_SF36.filter(like="_PreOp-3").iloc[:, 0],
    "VAS"  : cleanData_VAS.filter(like="_PreOp-3").iloc[:, 0],
    "SRS22": cleanData_SRS22.filter(like="_PreOp-3").iloc[:, 0]
}

mergedDF_PreOp3 = pd.concat(filteredData2, axis=1)
mergedDF_PreOp3

In [ ]:
createComparisonViolinPlots(mergedDF_PreOp3, "PreOp - 3")

again, the best ones are ODI, COMI and SF36 (just looking at the number of considered patients)

now I want to check the number of NaN values for each qestionnaire, for deciding which are the best for the analysis

In [ ]:
dataframe

In [ ]:
ODIpreopCount   = dataframe["ODI_Total_PreOp"].notna().sum()
ODI3monthsCount = dataframe["ODI_Total_3months"].notna().sum()
ODI6monthsCount = dataframe["ODI_Total_6months"].notna().sum()

COMIpreopCount   = dataframe["COMI_Total_PreOp"].notna().sum()
COMI3monthsCount = dataframe["COMI_Total_3months"].notna().sum()
COMI6monthsCount = dataframe["COMI_Total_6months"].notna().sum()

SF36preopCount   = dataframe["SF36_Pain_PreOp"].notna().sum()
SF363monthsCount = dataframe["SF36_Pain_3months"].notna().sum()
SF366monthsCount = dataframe["SF36_Pain_6months"].notna().sum()

SF36preopCount   = dataframe["SF36_MentalScore_PreOp"].notna().sum()
SF363monthsCount = dataframe["SF36_MentalScore_3months"].notna().sum()
SF366monthsCount = dataframe["SF36_MentalScore_6months"].notna().sum()

SF36preopCount   = dataframe["SF36_EnergyFatigue_PreOp"].notna().sum()
SF363monthsCount = dataframe["SF36_EnergyFatigue_3months"].notna().sum()
SF366monthsCount = dataframe["SF36_EnergyFatigue_6months"].notna().sum()

SF36preopCount  = dataframe["SF36_PhysicalScore_PreOp"].notna().sum()
SF363monthsCount = dataframe["SF36_PhysicalScore_3months"].notna().sum()
SF366monthsCount = dataframe["SF36_PhysicalScore_6months"].notna().sum()

SF36preopCount   = dataframe["SF36_GeneralHealth_PreOp"].notna().sum()
SF363monthsCount = dataframe["SF36_GeneralHealth_3months"].notna().sum()
SF366monthsCount = dataframe["SF36_GeneralHealth_6months"].notna().sum()


print(f"ODI  PreOp               : {ODIpreopCount}, ODI 3 months                : {ODI3monthsCount}, ODI 6 months                : {ODI6monthsCount}", "\n")
print(f"COMI PreOp               : {COMIpreopCount}, COMI 3 months               : {COMI3monthsCount}, COMI 6 months               : {COMI6monthsCount}", "\n")
print(f"SF36 Pain PreOp          : {SF36preopCount}, SF36 Pain 3 months          : {SF363monthsCount}, SF36 Pain 6 months          : {SF366monthsCount}")
print(f"SF36 Mental Score PreOp  : {SF36preopCount}, SF36 Mental Score 3 months  : {SF363monthsCount}, SF36 Mental Score 6 months  : {SF366monthsCount}")
print(f"SF36 Energy/Fatigue PreOp: {SF36preopCount}, SF36 Energy/Fatigue 3 months: {SF363monthsCount}, SF36 Energy/Fatigue 6 months: {SF366monthsCount}")
print(f"SF36 Physical Score PreOp: {SF36preopCount}, SF36 Physical Score 3 months: {SF363monthsCount}, SF36 Physical Score 6 months: {SF366monthsCount}")
print(f"SF36 General Health PreOp: {SF36preopCount}, SF36 General Health 3 months: {SF363monthsCount}, SF36 General Health 6 months: {SF366monthsCount}")

big difference between PreOp and 3months
around 400 between 3 and 6 months

check with MCD which one between 3 and 6 months in the most significant

From `Roland-Morris Disability Questionnaire, Oswestry Disability Index, and Quebec Back Pain Disability Scale: 
Which Has Superior Measurement Properties in Older Adults With Low Back Pain?`  

The results of the test-retest reliability showed sufficient reliability as ICCagreement values exceeded 0.70: 
- RMDQ, ICC(2,1) = 0.87 (95% confidence interval [CI]: 0.75, 0.94); 
- ODI, ICC(2,1) = 0.94 (95% CI: 0.88, 0.97); 
- QBPDS, ICC(2,1) = 0.95 (95% CI: 0.91, 0.98).      

From `Cross-cultural adaptation and validation of the Hungarian version
of the Core Outcome Measures Index for the back (COMI Back)`  

The ICC for the COMI total score was 0.92 (95 % CI 0.90–0.94).   
SEM for the COMI total score was 0.59 and thus the MDC95 % was calculated to be 1.63 points.

In [ ]:
def computeSEM(questionnaire: str) -> float:
    SDbaseline = dataframe.filter(regex = f"{questionnaire}_(Total|GeneralHealth)_PreOp").std().iloc[0]
    
    alpha = globals()[f"alpha{questionnaire}"]
    SEM = SDbaseline * np.sqrt(1 - alpha)
    
    return SEM

In [ ]:
def computeMDC95(questionnaire: str) -> float:
    SEM = computeSEM(questionnaire)
    MDC95 = SEM * 1.96 * 2**0.5
    
    return MDC95

In [ ]:
def createTargetVariables(questionnaire: str, MDC95: float) -> pd.DataFrame:
    questionnaireCols = dataframe.filter(regex = f"^{questionnaire}_.*(PreOp|3months|6months)$").columns
    questionnaireData = dataframe[questionnaireCols].copy()
    
    col_PreOp = questionnaireData.filter(regex="(Total|GeneralHealth)_PreOp$").columns[0]
    col_3m    = questionnaireData.filter(regex="(Total|GeneralHealth)_3months$").columns[0]
    col_6m    = questionnaireData.filter(regex="(Total|GeneralHealth)_6months$").columns[0]
    
    if questionnaire == "SF36":
        diffPreOp_3 = questionnaireData[col_3m] - questionnaireData[col_PreOp]
        diff3_6     = questionnaireData[col_6m] - questionnaireData[col_3m]
    
    elif questionnaire in ["ODI", "COMI"]:
        diffPreOp_3 = questionnaireData[col_PreOp] - questionnaireData[col_3m]
        diff3_6     = questionnaireData[col_3m] - questionnaireData[col_6m]

    targetColName_PreOp_3 = "isHealthier_PreOp_3months"
    targetColName_3_6     = "isHealthier_3months_6months"
   
    questionnaireData[targetColName_PreOp_3] = np.where(diffPreOp_3.isna(), np.nan, (diffPreOp_3 > MDC95).astype(float))
    questionnaireData[targetColName_PreOp_3] = questionnaireData[targetColName_PreOp_3].astype("Int64")
    
    questionnaireData[targetColName_3_6] = np.where(diff3_6.isna(), np.nan, (diff3_6 > MDC95).astype(float))
    questionnaireData[targetColName_3_6] = questionnaireData[targetColName_3_6].astype("Int64")

    return questionnaireData

In [ ]:
SEM_ODI = computeSEM("ODI")
MDC95_ODI = computeMDC95("ODI")
ODIdataframe = createTargetVariables("ODI", MDC95_ODI)

print(ODIdataframe["isHealthier_PreOp_3months"].value_counts())
print(ODIdataframe["isHealthier_3months_6months"].value_counts())

print(ODIdataframe["isHealthier_PreOp_3months"].value_counts().sum())
print(ODIdataframe["isHealthier_3months_6months"].value_counts().sum())

In [ ]:
SEM_COMI = computeSEM("COMI")
MDC95_COMI = computeMDC95("COMI")
COMIdataframe = createTargetVariables("COMI", MDC95_COMI)

print(COMIdataframe["isHealthier_PreOp_3months"].value_counts())
print(COMIdataframe["isHealthier_3months_6months"].value_counts())

print(COMIdataframe["isHealthier_PreOp_3months"].value_counts().sum())
print(COMIdataframe["isHealthier_3months_6months"].value_counts().sum())

In [ ]:
SEM_SF36 = computeSEM("SF36")
MDC95_SF36 = computeMDC95("SF36")
SF36dataframe = createTargetVariables("SF36", MDC95_SF36)

print(SF36dataframe["isHealthier_PreOp_3months"].value_counts())
print(SF36dataframe["isHealthier_3months_6months"].value_counts())

print(SF36dataframe["isHealthier_PreOp_3months"].value_counts().sum())
print(SF36dataframe["isHealthier_3months_6months"].value_counts().sum())

MCID

In [ ]:
MCID_SD_ODI = 0.5 * dataframe["ODI_Total_PreOp"].std()
print(f"Threshold MCID ODI (0.5 SD): {MCID_SD_ODI:.2f}")
print(f"Threshold MDC95 ODI: {MDC95_ODI:.2f}")
print(f"Threshold SEM ODI: {SEM_ODI:.2f}")

In [ ]:
MCID_SD_COMI = 0.5 * dataframe["COMI_Total_PreOp"].std()
print(f"Threshold MCID COMI (0.5 SD): {MCID_SD_COMI:.2f}")
print(f"Threshold MDC95 COMI: {MDC95_COMI:.2f}")
print(f"Threshold SEM COMI: {SEM_COMI:.2f}")

In [ ]:
MCID_SD_SF36 = 0.5 * dataframe["SF36_GeneralHealth_PreOp"].std()
print(f"Threshold MCID SF-36 (0.5 SD): {MCID_SD_SF36:.2f}")
print(f"Threshold MDC95 SF-36: {MDC95_SF36:.2f}")
print(f"Threshold SEM SF-36: {SEM_SF36:.2f}")

In [ ]:
def plotPieCharts(ODIdataframe, COMIdataframe, SF36dataframe):
    summary_df = pd.DataFrame({
        "ODI": ODIdataframe["isHealthier_PreOp_3months"].value_counts(),
        "COMI": COMIdataframe["isHealthier_PreOp_3months"].value_counts(),
        "SF36": SF36dataframe["isHealthier_PreOp_3months"].value_counts()
    }).reindex([0, 1], fill_value=0)

    labels = ['Below Threshold', 'Above Threshold (Healthier)']
    colors = ["#27ae60", "#7b3294"] 
    names = ["ODI", "COMI", "SF36"]
    titles = [
        "Clinical Success ODI (MDC95)", 
        "Clinical Success COMI (MDC95)", 
        "Clinical Success SF-36 (MDC95)"
    ]

    fig, axes = plt.subplots(1, 3, figsize=(18, 7))
    
    fig.suptitle("Clinical Success at 3 Months Post-Surgery (MDC95 Threshold)", fontsize=18, fontweight='bold', y=0.98)

    for i, ax in enumerate(axes):
        data = summary_df[names[i]]
        
        wedges, texts, autotexts = ax.pie(
            data, 
            labels=None, 
            autopct='%1.1f%%', 
            startangle=140, 
            colors=colors, 
            explode=(0.05, 0), 
            shadow=True,
            wedgeprops={'edgecolor': 'white', 'linewidth': 0.2}, 
            textprops={'fontsize': 12, 'fontweight': 'bold'} 
        )

        for autotext in autotexts:
            if autotexts.index(autotext) == 1 or autotexts.index(autotext) == 0:
                autotext.set_color('white')

        ax.set_aspect('equal')
        ax.axis('off') 
        
        ax.set_title(titles[i], fontsize=14, fontweight='bold', pad=20)

    fig.legend(
        wedges, 
        labels, 
        title="Clinical Success Status", 
        title_fontsize=12,
        loc="lower center", 
        bbox_to_anchor=(0.5, 0.05), 
        ncol=2, 
        fontsize=11,
        frameon=True, 
        shadow=True 
    )

    plt.tight_layout(rect=[0, 0.1, 1, 0.95])
    plt.savefig("Clinical_Success_Pie_Charts_MDC95.png", dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
plotPieCharts(ODIdataframe, COMIdataframe, SF36dataframe)

In [ ]:
def plotPieCharts(ODIdataframe, COMIdataframe, SF36dataframe):
    summary_df = pd.DataFrame({
        "ODI": ODIdataframe["isHealthier_3months_6months"].value_counts(),
        "COMI": COMIdataframe["isHealthier_3months_6months"].value_counts(),
        "SF36": SF36dataframe["isHealthier_3months_6months"].value_counts()
    }).reindex([0, 1], fill_value=0)

    labels = ['Below Threshold', 'Above Threshold (Healthier)']
    colors = ["#27ae60", "#7b3294"] 
    names = ["ODI", "COMI", "SF36"]
    titles = [
        "Clinical Success ODI (MDC95)", 
        "Clinical Success COMI (MDC95)", 
        "Clinical Success SF-36 (MDC95)"
    ]

    fig, axes = plt.subplots(1, 3, figsize=(18, 7))
    
    fig.suptitle("Clinical Success between 3 and 6 Months Post-Surgery (MDC95 Threshold)", fontsize=18, fontweight='bold', y=0.98)

    for i, ax in enumerate(axes):
        data = summary_df[names[i]]
        
        wedges, texts, autotexts = ax.pie(
            data, 
            labels=None, 
            autopct='%1.1f%%', 
            startangle=140, 
            colors=colors, 
            explode=(0.05, 0), 
            shadow=True,
            wedgeprops={'edgecolor': 'white', 'linewidth': 0.2}, 
            textprops={'fontsize': 12, 'fontweight': 'bold'} 
        )

        for autotext in autotexts:
            if autotexts.index(autotext) == 1 or autotexts.index(autotext) == 0:
                autotext.set_color('white')

        ax.set_aspect('equal')
        ax.axis('off') 
        
        ax.set_title(titles[i], fontsize=14, fontweight='bold', pad=20)

    fig.legend(
        wedges, 
        labels, 
        title="Clinical Success Status", 
        title_fontsize=12,
        loc="lower center", 
        bbox_to_anchor=(0.5, 0.05), 
        ncol=2, 
        fontsize=11,
        frameon=True, 
        shadow=True 
    )

    plt.tight_layout(rect=[0, 0.1, 1, 0.95])
    plt.savefig("Clinical_Success_Pie_Charts_MDC95_3-6months.png", dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
plotPieCharts(ODIdataframe, COMIdataframe, SF36dataframe)

definire funzione di calcolo del net benefit e partire con l'utilizzo di questa funzione per il calcolo degli iperparametri

TARGET --> ODI, COMI, SF36

In [ ]:
data1 = dataframe.loc[:, ~dataframe.columns.str.contains("ODI")].copy()
data2 = dataframe.loc[:, ~dataframe.columns.str.contains("COMI")].copy()
data3 = dataframe.loc[:, ~dataframe.columns.str.contains("SF36")].copy()

In [ ]:
ODIdataframe  = pd.concat([ODIdataframe, data1], axis=1)
COMIdataframe = pd.concat([COMIdataframe, data2], axis=1)
SF36dataframe = pd.concat([SF36dataframe, data3], axis=1)

In [ ]:
ODIdataframe = ODIdataframe.dropna(subset=["ODI_Total_PreOp", "ODI_Total_3months"])
ODIdataframe = ODIdataframe.drop(ODIdataframe.filter(regex=r"^(?!.*ODI)((?!.*isHealthier_PreOp_3months)).*(3months|6months|12months|24months|60months).*").columns, axis=1)
ODIdataframe = ODIdataframe.drop(ODIdataframe.filter(regex="_date").columns, axis=1)
ODIdataframe = ODIdataframe.drop(columns = ["first_operator_1", "patient_state", "state", "intervention_side_1", "ICD_code_INTRAOP_1", "ICD_code_POSTOP_1"])

print("Number of rows:", len(ODIdataframe))

In [ ]:
ODIdataframe

checking ICD_1 and ICD_PreOP distributions

In [ ]:
frequencies = dataframe['ICD_1'].value_counts()
perc = dataframe['ICD_1'].value_counts(normalize=True) * 100

dfFreq = pd.DataFrame({'Counts': frequencies, 'Percentage %': perc.round(2)})
print(dfFreq.head(10))

In [ ]:
topN = 20
topCodes = dataframe["ICD_1"].value_counts().nlargest(topN).index

dfTopCodes = dataframe[dataframe["ICD_1"].isin(topCodes)]

plt.figure(figsize=(12, 6))
sns.countplot(
    data=dfTopCodes,
    x="ICD_1",
    order=topCodes, 
    color="#1e894b"  
)

plt.title(f"Top {topN} ICD_1 Codes", fontsize=14)
plt.xlabel("ICD_1 Code", fontsize=12)
plt.ylabel("Frequency (Count)", fontsize=12)

plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
topN = 20
topCodes = dataframe["ICD_code_PREOP_1"].value_counts().nlargest(topN).index

dfTopCodes = dataframe[dataframe["ICD_code_PREOP_1"].isin(topCodes)]

plt.figure(figsize=(12, 6))
sns.countplot(
    data=dfTopCodes,
    x="ICD_code_PREOP_1",
    order=topCodes, 
    color="#1e894b"  
)

plt.title(f"Top {topN} ICD_code_PREOP_1 Codes", fontsize=14)
plt.xlabel("ICD_code_PREOP_1 Code", fontsize=12)
plt.ylabel("Frequency (Count)", fontsize=12)

plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
uniqueValues = pd.Series(dataframe["ICD_1"].unique())
print(uniqueValues.sort_values().to_string(index=False))

In [ ]:
levels = dataframe["levels_1"]
levels

LEVELS 

In [ ]:
levels = pd.Series(levels)
levels = levels.str.upper()

In [ ]:
for pos,element in enumerate(levels):
    if pd.isna(element): continue
    newElement = ""
    for part in element.split(' '):
        if not re.search(r'LOMB|SACR|ILIAC|PELV|SOMA|PEDIC', part):
            if part == "ILEO": part = "Y0"
            newElement += part + ' '

    if not newElement:
        if "LOMB"  in element: newElement += "L0 "
        if "SACR"  in element: newElement += "S0 "
        if "ILIAC" in element: newElement += "Y0 "
        
    levels[pos] = newElement[:-1]

Hyperparameter optimization for ODI (standard)

In [ ]:
class Vertebra():
    def __init__(self, type: str, number: str):
        self.type : Vertebra.Type = Vertebra.Type.fromStr(type)
        self.number = int(number) if number != '0' else None
    
    class Type(Enum):
        CERVICAL = "C"
        THORACIC = "T"
        LUMBAR   = "L"
        SACRAL   = "S"
        ILEO     = "Y"

        def fromStr(s: str) -> "Vertebra.Type":
            s = s.upper()
            if s == 'D': return Vertebra.Type.THORACIC
            return Vertebra.Type(s)
        
        def getAmount(self) -> int:
            match self:
                case Vertebra.Type.CERVICAL: return 7
                case Vertebra.Type.THORACIC: return 12
                case Vertebra.Type.LUMBAR:   return 5
                case Vertebra.Type.SACRAL:   return 5
                case Vertebra.Type.ILEO:     return 0
        
        def countOffset(self) -> int:
            sum = 0
            for t in Vertebra.Type:
                if self == t: 
                    return sum
                sum += t.getAmount()

    @staticmethod
    def parsingLevels1(level: str) -> tuple["Vertebra", "Vertebra"]:
        start, stop = level.split("-")

        # get position
        posStart  = start[1:]
        posStop  = stop[1:]

        typeStart = start[0]
        typeStop  = stop[0]

        return Vertebra(typeStart, posStart), Vertebra(typeStop, posStop)
    
    
    @staticmethod
    def parsingLevels2(level: str) -> list[tuple["Vertebra", "Vertebra"]]:
        parts = level.split(" ")
        vertebrae = []
        for part in parts:
            if part == "ILEO":
                vertebrae.append((Vertebra(part, "0"), ) * 2)
                continue
            vertebrae.append(Vertebra.parsingLevels1(part))

        return vertebrae
    
    # III L VL VL IS
    @staticmethod
    def parsingLevels4(level: str) -> list["Vertebra"]:
        numbers = {"I": 1, "II": 2, "III": 3, "IV": 4, "V": 5, "VI": 6, "VII": 7, "VIII": 8, "IX": 9, "X": 10, "XI": 11, "XII": 12}
        parts = level.split(" ")
        vertebrae = []
        currentNum = 0
        
        for part in parts:
            if part == "Y0":
                vertebrae.append(Vertebra(*part))
                continue
            
            if currentNum:
                vertebrae.append(Vertebra(part, currentNum))
                currentNum = 0
                continue
            
            try: # caso in cui non c'è lo spazio in mezzo tra il numero e la lettera della vertebra
                vertebrae.append(Vertebra(part[-1], numbers[part[:-1]])) 

            except (KeyError, ValueError): 
                currentNum = numbers[part]
    
        return vertebrae
    
    # L531S1
    @staticmethod
    def parsingLevels5(level: str) -> list["Vertebra"]:
        vertebrae = []
        number = ""
        currentType = None

        for element in level:
            if element.isdigit(): number += element
                
            else:
                if currentType is not None: # c'è già una vertebra fatta che qui finisce
                    vertebrae.append(Vertebra(currentType, number))
                        
                currentType = element
                number = ""
        
        vertebrae.append(Vertebra(currentType, number)) # sennò mi perdo l'ultima vertebra
            
        return vertebrae
    
    # L5S1 L3-S1 C5C6 C6C7
    @staticmethod
    def parsingLevels6(level: str) -> list["Vertebra"]:
        parts = level.split(' ')
        vertebrae = []
        
        for element in parts:
            if '-' in element: vertebrae.append(Vertebra.parsingLevels1(element))
            else: vertebrae.append(Vertebra.parsingLevels5(element))
        
        return vertebrae
    
    # TODO: per il MEGA PARSER 
    # splittare per spazi
    # tirare via gli ileo
    # numeri romani
    # con trattino
    # senza trattino

    def parsingLevels(levels: pd.Series) -> list[list["Vertebra"]]:
        vertebrae: list[list[Vertebra]] = []
        for element in levels:

            if pd.isna(element):
                vertebrae.append([])
                continue

            if re.search(r"I|II|III|IV|V|VI|VII|VIII|IX|X|XI|XII", element):
                vertebrae.append(Vertebra.parsingLevels4(element))
                continue

            parts = element.split(' ')
            vertebrae.append([])

            for part in parts:
                if '-' in part: 
                    vertebrae[-1].extend(Vertebra.parsingLevels1(part)) 
                else:
                    vertebrae[-1].extend(Vertebra.parsingLevels5(part))

        return vertebrae
    
    def computeAbsPos(self) -> int:
        return self.number + self.type.countOffset()

    def __repr__(self):
        return f"{self.type.value}{self.number if self.number is not None else '?'}"

In [ ]:
Vertebra.Type.CERVICAL.countOffset()
Vertebra("L", "4").computeAbsPos()

In [ ]:
vertebrae = Vertebra.parsingLevels(levels)
vertebrae

In [ ]:
len(Vertebra.Type)
dataframe.shape

In [ ]:
results = {t: [0] * dataframe.shape[0] for t in Vertebra.Type} # 0 in ogni riga per ogni vertebra type 

for pos,row in enumerate(vertebrae):
    for vertebra in row:
        results[vertebra.type][pos] = 1 

for colName, value in results.items():
    dataframe[colName.name] = value

dataframe

In [ ]:
distances = []
for row in vertebrae:
    minNum = None
    maxNum = None
    difference = None
    
    for vertebra in row:
        if vertebra is None or vertebra.number is None:
            continue
        
        absPos = vertebra.computeAbsPos()

        if not minNum or absPos < minNum:
            minNum = absPos

        if not maxNum or absPos > maxNum:
            maxNum = absPos
    
    if minNum and maxNum: difference = maxNum - minNum
    distances.append(difference)

dataframe['distanceBetweenVertebrae'] = distances
dataframe

Remove some columns 

In [ ]:
colsICDintraop = dataframe.filter(regex = "ICD_code_INTRAOP").columns
colsICDpostop  = dataframe.filter(regex = "ICD_code_POSTOP").columns

cols12m = dataframe.filter(regex = "_12months").columns
cols24m = dataframe.filter(regex = "_24months").columns
cols60m = dataframe.filter(regex = "_60months").columns

dataframe = dataframe.drop(columns = colsICDintraop)
dataframe = dataframe.drop(columns = colsICDpostop)
dataframe = dataframe.drop(columns = cols12m)
dataframe = dataframe.drop(columns = cols24m)
dataframe = dataframe.drop(columns = cols60m)

dataframe

Try with 6 columns

In [ ]:
diseaseGroupCount = {}
for row in dataframe["ICD_code_PREOP_1"]:
    if pd.isna(row):
        continue

    if row[0] == "V":
        group = "Follow-up and revision codes"
        diseaseGroupCount[group] = diseaseGroupCount.get(group, 0) + 1 
        continue
    
    # guardo solo le prime tre cifre (anche se non hanno il . vanno bene)
    firstNum = int(row[:3])

    if firstNum in [343, 344, 723, 724, 747]:
        # paralisi, cervicalgia, lombalgia, dorsalgia, postumi stenosi canale vertebrale
        group = "Clinical manifestations and symptoms"

    elif firstNum in [718, 732, 733, 737, 738, 754, 756, 805, 806, 839, 905]:
        # spondilolistesi, frattura patologica, malattia di Scheuermann, scoliosi, malformazione colonna, lussazione, frattura colonna, postumi fratture, altre deformazioni acquisite dell’apparato muscoloscheletrico
        group = "Mechanical, traumatic, and morphological alterations"
    
    elif firstNum in [336, 349, 353, 715, 720, 721, 722, 727]:
        # artrosi, spondilite, spondilosi, discoartrosi, sinoviti/tenosinoviti, malattie del midollo, disturbi delle radici e dei plessi nervosi, disturbi sistema nervoso
        group = "Degenerative, inflammatory, and neurological processes"
        
    elif firstNum in [170, 171, 198, 203, 213, 225, 228, 237, 730, 996, 998]:
        # osteosarcoma, metastasi ossee, mieloma multiplo, tumori benigni encefalo, emangioma, neurofibromatosi, infezioni ossee, 996 complicazioni infezioni, 998 infezione, tumori benigni delle ossa, tumori maligni del connettivo e di altri tessuti molli
        group = "Disease_Neoplastic_pathologies_and_infections"
    
    else:
        group = "Disease_Others"
        # 388 disturbi dell'orecchio, 602 prostata, 277 disturbi del metabolsimo, 332 parkinson, 340 sclerosi multipla
        
    diseaseGroupCount[group] = diseaseGroupCount.get(group, 0) + 1 
    #^^^ get() per default value se non c'è già la chiave (0) + 1 ogni volta che ne trova uno (se la chiave c'è già, valore)

diseaseGroupCount

In [ ]:
diseaseGroup = []

for row in dataframe["ICD_code_PREOP_1"]:
    if pd.isna(row):
        diseaseGroup.append(pd.NA)
        continue

    if row[0] == "V":
        group = "Disease_Followup_revision"
        diseaseGroup.append(group)
        continue

    firstNum = int(row[:3])

    if firstNum in [343, 344, 723, 724, 747]:
        group = "Disease_Clinical_manifestations_symptoms"

    elif firstNum in [718, 732, 733, 737, 738, 754, 756, 805, 806, 839, 905]:
        group = "Disease_Mechanical_traumatic_morphological_alterations"
    
    elif firstNum in [336, 349, 353, 715, 720, 721, 722, 727]:
        group = "Disease_Degenerative_inflammatory_neurological_processes"
        
    elif firstNum in [170, 171, 198, 203, 213, 225, 228, 237, 730, 996, 998]:
        group = "Disease_Neoplastic_pathologies_and_infections"
    
    else:
        group = "Disease_Others"

    diseaseGroup.append(group)

diseaseGroup

In [ ]:
columns = pd.get_dummies(diseaseGroup, dtype = int)
dataframe = pd.concat([dataframe, columns], axis=1)
dataframe

In [ ]:
targetCols = [col for col in dataframe.columns if col.startswith('Disease_')]
totals = dataframe[targetCols].sum()

print(totals)

ICD codes for procedures -> Raggruppamento per Finalità Clinica  



In [ ]:
uniqueValues = pd.Series(dataframe["ICD_1"].unique())
print(uniqueValues.sort_values().to_string(index=False))

In [ ]:
procedureGroupCount = {}
for row in dataframe["ICD_1"]:
    if pd.isna(row):
        continue
    
   # per i codici di procedura guardo solo le prime due cifre
    firstNum = int(str(row).split('.')[0])

    if firstNum in [0, 1, 2, 3, 4]:
        group = "Procedure_Nervous_system"
    elif firstNum in [76, 77, 78, 79, 80, 81, 83, 84, 86]:
        group = "Procedure_Musculoskeletal_integumentary_system"
    else:
        group = "Procedure_Others"
        
    procedureGroupCount[group] = procedureGroupCount.get(group, 0) + 1 
    
procedureGroupCount

In [ ]:
procedureGroup = []
for row in dataframe["ICD_1"]:
    if pd.isna(row):
        procedureGroup.append(pd.NA)
        continue
    
    firstNum = int(str(row).split('.')[0])

    if firstNum in [0, 1, 2, 3, 4]:
        group = "Procedure_Nervous_system"
    elif firstNum in [76, 77, 78, 79, 80, 81, 83, 84, 86]:
        group = "Procedure_Musculoskeletal_integumentary_system"
    else:
        group = "Procedure_Others"
        
    procedureGroup.append(group)
    
procedureGroup

In [ ]:
columns = pd.get_dummies(procedureGroup, dtype = int)
dataframe = pd.concat([dataframe, columns], axis=1)
dataframe

In [ ]:
targetCols = [col for col in dataframe.columns if col.startswith('Procedure_')]
totals = dataframe[targetCols].sum()

print(totals)

In [ ]:
print(dataframe.columns.tolist())

In [ ]:
corrDF = dataframe[['CERVICAL', 'THORACIC', 'LUMBAR', 'SACRAL', 'ILEO', 'distanceBetweenVertebrae', 'Disease_Clinical_manifestations_symptoms', 'Disease_Degenerative_inflammatory_neurological_processes', 'Disease_Followup_revision', 'Disease_Mechanical_traumatic_morphological_alterations', 'Disease_Neoplastic_pathologies_and_infections', 'Disease_Others', 'Procedure_Musculoskeletal_integumentary_system', 'Procedure_Nervous_system', 'Procedure_Others']]

corrDF.columns = [col.replace('category_', '').replace('_', ' ').title() for col in corrDF.columns]
corrMatrix = corrDF.corr(numeric_only=True)

plt.figure(figsize=(10, 8))
sns.heatmap(
    corrMatrix, 
    fmt=".2f", 
    cmap='PRGn', 
    linewidths=0.5, 
    center=0
)

plt.title("Correlation Analysis (new variables)", fontsize=14, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0) 

plt.tight_layout()

plt.show()

In [ ]:
dataframe = dataframe.drop(columns = ["levels_1", "ICD_1", "ICD_code_PREOP_1"])

In [ ]:
colsToDrop6m = dataframe.filter(like = "_6months").columns
dataframe = dataframe.drop(columns = colsToDrop6m)

colsSF36 = dataframe.filter(regex = r'^(?=.*SF36)(?!.*(?:GeneralHealth|completion_date))').columns
dataframe = dataframe.drop(columns = colsSF36)

colsSRS22 = dataframe.filter(regex = "^(?=.*SRS22)(?!.*Total)").columns
dataframe = dataframe.drop(columns = colsSRS22)

colsToDrop3mVAS = dataframe.filter(regex = "Vas.*_3months").columns
dataframe = dataframe.drop(columns = colsToDrop3mVAS)

colsToDrop3mSRS22 = dataframe.filter(regex = "SRS22.*_3months").columns
dataframe = dataframe.drop(columns = colsToDrop3mSRS22)

colsToDropCompletionDate = dataframe.filter(regex = "completion_date").columns
dataframe = dataframe.drop(columns = colsToDropCompletionDate)

colsToDropImplants = dataframe.filter(regex = "implants_0no_1yes[2|3]").columns
dataframe = dataframe.drop(columns = colsToDropImplants)

dataframe

now I use 'surgery_date_1' and 'discharge_date' to create the column 'days_at_hospital'

In [ ]:
dataframe['surgery_date_1'] = pd.to_datetime(dataframe['surgery_date_1'])
dataframe['discharge_date'] = pd.to_datetime(dataframe['discharge_date'])

dataframe['days_at_hospital'] = (dataframe['discharge_date'] - dataframe['surgery_date_1']).dt.days

dataframe = dataframe.drop(columns = ['surgery_date_1', 'discharge_date'])
dataframe

using 'state' to create a binary column 'isActive'

In [ ]:
dataframe['isActive'] = np.where(dataframe['state'] == 'active', 1, 0)

dataframe = dataframe.drop(columns = ['state', 'patient_state'])
dataframe

Drop these columns because 99% of their data has the same value

In [ ]:
dataframe = dataframe.drop(columns = ['anesthesia_1', 'clean_intervention_1'])
dataframe

In [ ]:
print(dataframe["current_step"].unique())

In [ ]:
step_mapping = {
    'Intraop-Discharge': 0,
    '3 months': 3,
    '6 months': 6,
    '12 months': 12,
    '24 months': 24,
    '5 years': 60
}

# Crea la nuova colonna numerica applicando la mappa
dataframe['current_step_months'] = dataframe['current_step'].map(step_mapping)
dataframe = dataframe.drop(columns = ['current_step'])
dataframe

In [ ]:
print(dataframe["intervention_side_1"].unique())

In [ ]:
dataframe['isRight_side'] = dataframe['intervention_side_1'].isin(['Right', 'Bilateral']).astype(float)
dataframe['isLeft_side']  = dataframe['intervention_side_1'].isin(['Left', 'Bilateral']).astype(float)

missingMask = dataframe['intervention_side_1'].isna()
dataframe.loc[missingMask, ['isRight_side', 'isLeft_side']] = np.nan

dataframe = dataframe.drop('intervention_side_1', axis=1)
dataframe

In [ ]:
print(dataframe["category"].unique())

In [ ]:
macro_groups = {
    'Lumbar arthrodesis': 'Lumbar',
    'Lumbar herniation': 'Lumbar',
    'Lumbar decompression': 'Lumbar',
    'Cervical arthrodesis': 'Cervical',
    'Cervical herniation': 'Cervical',
    'Degenerative deformity': 'Deformity',
    'Idiopatic deformity': 'Deformity',
    'Neuromuscular scoliosis': 'Deformity',
    'Kyphoplasty': 'Other',
    'Vertebral cancer': 'Other'
}

dataframe['category'] = dataframe['category'].map(macro_groups)
dataframe = pd.get_dummies(dataframe, columns=['category'], dtype=int)

dataframe

In [ ]:
dataframe = dataframe.drop('first_operator_1', axis=1)
dataframe

In [ ]:
dataframe.columns.tolist()

In [ ]:
columns = ['asa_class_1',
    'image_intensifier_0no_1yes_1',
    'xray_radiography_0no_1yes_1',
    'biopsy_histological_0no_1yes_1',
    'culture_0no_1yes_1',
    'implants_0no_1yes_1',
    'explanted_implants_0no_1yes_1',
    'fixation_device_0no_1yes_1',
    'tourniquet_0no_1yes_1',
    'sex_0M_1F',
    'age_at_enrollment',
    'BMI_Total_PreOp',
    'Blood_Loss',
    'Blood_Transfusion',
    'CERVICAL',
    'THORACIC',
    'LUMBAR',
    'SACRAL',
    'ILEO',
    'distanceBetweenVertebrae',
    'Disease_Clinical_manifestations_symptoms',
    'Disease_Degenerative_inflammatory_neurological_processes',
    'Disease_Followup_revision',
    'Disease_Mechanical_traumatic_morphological_alterations',
    'Disease_Neoplastic_pathologies_and_infections',
    'Disease_Others',
    'Procedure_Musculoskeletal_integumentary_system',
    'Procedure_Nervous_system',
    'Procedure_Others',
    'days_at_hospital',
    'isActive',
    'current_step_months',
    'isRight_side',
    'isLeft_side',
    'category_Cervical',
    'category_Deformity',
    'category_Lumbar',
    'category_Other'
]

corr_matrix = dataframe[columns].corr()

plt.figure(figsize=(22, 18))
sns.heatmap(corr_matrix, 
            cmap='PRGn', 
            vmax=1.0,          
            vmin=-1.0,         
            center=0,          
            square=True,      
            linewidths=.5,     
            cbar_kws={"shrink": .7}, 
            annot=False)    

plt.title('Correlation Matrix (with all variables, except questionnaires)', fontsize=24, pad=20)
plt.tight_layout()
plt.savefig("Variables_Correlation.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
dataframe

In [ ]:
def createTargetVariablesFinal(questionnaire: str, MDC95: float) -> pd.DataFrame:
    questionnaireData = dataframe.copy()
    
    col_PreOp = questionnaireData.filter(regex=f"^{questionnaire}_.*(Total|GeneralHealth)_PreOp$").columns[0]
    col_3m    = questionnaireData.filter(regex=f"^{questionnaire}_.*(Total|GeneralHealth)_3months$").columns[0]
    
    if questionnaire == "SF36":
        diffPreOp_3 = questionnaireData[col_3m] - questionnaireData[col_PreOp]
    
    elif questionnaire in ["ODI", "COMI"]:
        diffPreOp_3 = questionnaireData[col_PreOp] - questionnaireData[col_3m]

    targetColName_PreOp_3 = f"overuse_PreOp_3months_{questionnaire}"
   
    questionnaireData[targetColName_PreOp_3] = np.where(diffPreOp_3.isna(), np.nan, (diffPreOp_3 <= MDC95).astype(float))
    questionnaireData[targetColName_PreOp_3] = questionnaireData[targetColName_PreOp_3].astype("Int64")

    return questionnaireData

In [ ]:
ODIdataframe  = createTargetVariablesFinal("ODI", MDC95_ODI)
COMIdataframe = createTargetVariablesFinal("COMI", MDC95_COMI)
SF36dataframe = createTargetVariablesFinal("SF36", MDC95_SF36)

In [ ]:
ODIdataframe

In [ ]:
print(ODIdataframe["overuse_PreOp_3months_ODI"].value_counts())
print(COMIdataframe["overuse_PreOp_3months_COMI"].value_counts())
print(SF36dataframe["overuse_PreOp_3months_SF36"].value_counts())

In [ ]:
ODIdataframe.drop(columns = ODIdataframe.filter(regex = "COMI|SF36").columns, inplace=True)
COMIdataframe.drop(columns = COMIdataframe.filter(regex = "ODI|SF36").columns, inplace=True)
SF36dataframe.drop(columns = SF36dataframe.filter(regex = "ODI|COMI").columns, inplace=True)

In [ ]:
ODIdataframe.drop(columns = ["ODI_Total_3months"], inplace=True)
COMIdataframe.drop(columns = ["COMI_Total_3months"], inplace=True)
SF36dataframe.drop(columns = ["SF36_GeneralHealth_3months"], inplace=True)

In [ ]:
ODIdataframe

ML MODELS

In [ ]:
def prepareData(questionnaire: str):
    if questionnaire == "ODI":
        dataframe = ODIdataframe.copy()
        target = "overuse_PreOp_3months_ODI"
    elif questionnaire == "COMI":
        dataframe = COMIdataframe.copy()
        target = "overuse_PreOp_3months_COMI"
    elif questionnaire == "SF36":
        dataframe = SF36dataframe.copy()
        target = "overuse_PreOp_3months_SF36"
    
    dataframe = dataframe.dropna(subset=[target])
    features = [col for col in dataframe.columns if col != target]
    x = dataframe[features]
    y = dataframe[target]

    xTrain, xTest, yTrain, yTest = train_test_split(x, y, test_size=0.2, random_state=883)

    imputer = SimpleImputer(strategy='median')
    scaler = StandardScaler()
    
    xTrainScaled = scaler.fit_transform(imputer.fit_transform(xTrain))
    xTestScaled = scaler.transform(imputer.transform(xTest))
    
    return xTrainScaled, xTestScaled, yTrain, yTest

In [ ]:
def computeAdditionalMetrics(questionnaire: str, yTrue, yPred, yProb):
    balancedAccuracy = balanced_accuracy_score(yTrue, yPred)
    overallF1 = f1_score(yTrue, yPred, average='macro')
    brierScore = brier_score_loss(yTrue, yProb)
    aucScore = roc_auc_score(yTrue, yProb)

    return balancedAccuracy, overallF1, brierScore, aucScore

LOGISTIC REGRESSION

In [ ]:
def logisticRegression(questionnaire: str) :
    xTrainScaled, xTestScaled, yTrain, yTest = prepareData(questionnaire)
    
    logisticModel = LogisticRegression(random_state=883)
    logisticModel.fit(xTrainScaled, yTrain)

    prediction = logisticModel.predict(xTestScaled)
    yProbs = logisticModel.predict_proba(xTestScaled)[:, 1]

    return logisticModel, xTestScaled, yTest, prediction, yProbs, confusion_matrix(yTest, prediction), classification_report(yTest, prediction)

In [ ]:
logisticModelODI, xTestScaledODI, yTestODI, predictionODI, yProbODI, confusionMatrixODI, reportODI = logisticRegression("ODI")

fig, ax = plt.subplots(1, 2, figsize=(16, 6))

confusionMatrix = confusion_matrix(yTestODI, predictionODI)
sns.heatmap(confusionMatrix, annot=True, fmt='d', cmap='Greens', cbar=False, ax=ax[0])
ax[0].set_xlabel('Predicted Label')
ax[0].set_ylabel('True Label')
ax[0].set_title('Confusion Matrix')

RocCurveDisplay.from_estimator(
    logisticModelODI, 
    xTestScaledODI, 
    yTestODI, 
    name="Logistic Regression",
    ax=ax[1],
    color='forestgreen'
)
ax[1].plot([0, 1], [0, 1], "k--", label="Chance Level (AUC = 0.5)")
ax[1].set_title("ROC Curve: Predicting Overuse (ODI)", fontsize=14)
ax[1].legend(loc='lower right') 

plt.tight_layout() 
plt.show()

print("\n--- Classification Report ---")
print(reportODI)

balancedAccuracy, overallF1, brierScore, aucScore = computeAdditionalMetrics("ODI", yTestODI, predictionODI, yProbODI)
print (f"Balanced Accuracy: {balancedAccuracy:.4f}")
print (f"Overall F1 Score: {overallF1:.4f}")
print (f"Brier Score: {brierScore:.4f}")
print (f"AUC Score: {aucScore:.4f}")

In [ ]:
def optimizedLogisticRegression(questionnaire: str):
    xTrainScaled, xTestScaled, yTrain, yTest = prepareData(questionnaire)

    paramGrid = [
        {
            'solver': ['liblinear'],
            'penalty': ['l1', 'l2'],
            'C': [0.001, 0.01, 0.1, 1, 10, 100]
        }
    ]

    gridSearch = GridSearchCV(
        LogisticRegression(random_state=883), 
        paramGrid, 
        cv=5, 
        scoring='roc_auc', 
        n_jobs=-1
    )

    gridSearch.fit(xTrainScaled, yTrain)
    logisticModel = gridSearch.best_estimator_
    prediction = logisticModel.predict(xTestScaled)
    yProbs = logisticModel.predict_proba(xTestScaled)[:, 1]

    print(f"Best parameters for {questionnaire}: {gridSearch.best_params_}")

    return logisticModel, xTestScaled, yTest, prediction, yProbs, confusion_matrix(yTest, prediction), classification_report(yTest, prediction)

In [ ]:
LogisticModelODIopt, xTestScaledODIopt, yTestODIopt, predictionODIopt, yProbsODIopt, confusionMatrixODIopt, reportODIopt = optimizedLogisticRegression("ODI")

fig, ax = plt.subplots(1, 2, figsize=(16, 6))

confusionMatrix = confusion_matrix(yTestODIopt, predictionODIopt)
sns.heatmap(confusionMatrix, annot=True, fmt='d', cmap='Greens', cbar=False, ax=ax[0])
ax[0].set_xlabel('Predicted Label')
ax[0].set_ylabel('True Label')
ax[0].set_title('Confusion Matrix')

RocCurveDisplay.from_estimator(
    LogisticModelODIopt, 
    xTestScaledODIopt, 
    yTestODIopt, 
    name="Logistic Regression",
    ax=ax[1],
    curve_kwargs={'color': 'forestgreen'}
)
ax[1].plot([0, 1], [0, 1], "k--", label="Chance Level (AUC = 0.5)")
ax[1].set_title("ROC Curve: Predicting Overuse (ODI)", fontsize=14)
ax[1].legend(loc='lower right') 

plt.tight_layout() 
plt.show()

print("\n--- Classification Report ---")
print(reportODIopt)

balancedAccuracy, overallF1, brierScore, aucScore = computeAdditionalMetrics("ODI", yTestODIopt, predictionODIopt, yProbsODIopt)
print (f"Balanced Accuracy: {balancedAccuracy:.4f}")
print (f"Overall F1 Score: {overallF1:.4f}")
print (f"Brier Score: {brierScore:.4f}")
print (f"AUC Score: {aucScore:.4f}")

In [ ]:
logisticModelCOMI, xTestScaledCOMI, yTestCOMI, predictionCOMI, yProbCOMI, confusionMatrixCOMI, reportCOMI = logisticRegression("COMI")

fig, ax = plt.subplots(1, 2, figsize=(16, 6))

confusionMatrix = confusion_matrix(yTestCOMI, predictionCOMI)
sns.heatmap(confusionMatrix, annot=True, fmt='d', cmap='Greens', cbar=False, ax=ax[0])
ax[0].set_xlabel('Predicted Label')
ax[0].set_ylabel('True Label')
ax[0].set_title('Confusion Matrix')

RocCurveDisplay.from_estimator(
    logisticModelCOMI, 
    xTestScaledCOMI, 
    yTestCOMI, 
    name="Logistic Regression",
    ax=ax[1],
    color='forestgreen'
)
ax[1].plot([0, 1], [0, 1], "k--", label="Chance Level (AUC = 0.5)")
ax[1].set_title("ROC Curve: Predicting Overuse (COMI)", fontsize=14)
ax[1].legend(loc='lower right') 

plt.tight_layout() 
plt.show()

print("\n--- Classification Report ---")
print(reportCOMI)

balancedAccuracy, overallF1, brierScore, aucScore = computeAdditionalMetrics("COMI", yTestCOMI, predictionCOMI, yProbCOMI)
print (f"Balanced Accuracy: {balancedAccuracy:.4f}")
print (f"Overall F1 Score: {overallF1:.4f}")
print (f"Brier Score: {brierScore:.4f}")
print (f"AUC Score: {aucScore:.4f}")

In [ ]:
LogisticModelCOMIopt, xTestScaledCOMIopt, yTestCOMIopt, predictionCOMIopt, yProbCOMIopt, confusionMatrixCOMIopt, reportCOMIopt = optimizedLogisticRegression("COMI")

fig, ax = plt.subplots(1, 2, figsize=(16, 6))

confusionMatrix = confusion_matrix(yTestCOMIopt, predictionCOMIopt)
sns.heatmap(confusionMatrix, annot=True, fmt='d', cmap='Greens', cbar=False, ax=ax[0])
ax[0].set_xlabel('Predicted Label')
ax[0].set_ylabel('True Label')
ax[0].set_title('Confusion Matrix')

RocCurveDisplay.from_estimator(
    LogisticModelCOMIopt, 
    xTestScaledCOMIopt, 
    yTestCOMIopt, 
    name="Logistic Regression",
    ax=ax[1],
    curve_kwargs={'color': 'forestgreen'}
)
ax[1].plot([0, 1], [0, 1], "k--", label="Chance Level (AUC = 0.5)")
ax[1].set_title("ROC Curve: Predicting Overuse (COMI)", fontsize=14)
ax[1].legend(loc='lower right') 

plt.tight_layout() 
plt.show()

print("\n--- Classification Report ---")
print(reportCOMIopt)

balancedAccuracy, overallF1, brierScore, aucScore = computeAdditionalMetrics("COMI", yTestCOMIopt, predictionCOMIopt, yProbCOMIopt)
print (f"Balanced Accuracy: {balancedAccuracy:.4f}")
print (f"Overall F1 Score: {overallF1:.4f}")
print (f"Brier Score: {brierScore:.4f}")
print (f"AUC Score: {aucScore:.4f}")

In [ ]:
logisticModelSF36, xTestScaledSF36, yTestSF36, predictionSF36, yProbSF36, confusionMatrixSF36, reportSF36 = logisticRegression("SF36")

fig, ax = plt.subplots(1, 2, figsize=(16, 6))

confusionMatrix = confusion_matrix(yTestSF36, predictionSF36)
sns.heatmap(confusionMatrix, annot=True, fmt='d', cmap='Greens', cbar=False, ax=ax[0])
ax[0].set_xlabel('Predicted Label')
ax[0].set_ylabel('True Label')
ax[0].set_title('Confusion Matrix')

RocCurveDisplay.from_estimator(
    logisticModelSF36, 
    xTestScaledSF36, 
    yTestSF36, 
    name="Logistic Regression",
    ax=ax[1],
    color='forestgreen'
)
ax[1].plot([0, 1], [0, 1], "k--", label="Chance Level (AUC = 0.5)")
ax[1].set_title("ROC Curve: Predicting Overuse (SF36)", fontsize=14)
ax[1].legend(loc='lower right') 

plt.tight_layout() 
plt.show()

print("\n--- Classification Report ---")
print(reportSF36)

balancedAccuracy, overallF1, brierScore, aucScore = computeAdditionalMetrics("SF36", yTestSF36, predictionSF36, yProbSF36)
print (f"Balanced Accuracy: {balancedAccuracy:.4f}")
print (f"Overall F1 Score: {overallF1:.4f}")
print (f"Brier Score: {brierScore:.4f}")
print (f"AUC Score: {aucScore:.4f}")

In [ ]:
LogisticModelSF36opt, xTestScaledSF36opt, yTestSF36opt, predictionSF36opt, yProbSF36opt, confusionMatrixSF36opt, reportSF36opt = optimizedLogisticRegression("SF36")

fig, ax = plt.subplots(1, 2, figsize=(16, 6))

confusionMatrix = confusion_matrix(yTestSF36opt, predictionSF36opt)
sns.heatmap(confusionMatrix, annot=True, fmt='d', cmap='Greens', cbar=False, ax=ax[0])
ax[0].set_xlabel('Predicted Label')
ax[0].set_ylabel('True Label')
ax[0].set_title('Confusion Matrix')

RocCurveDisplay.from_estimator(
    LogisticModelSF36opt, 
    xTestScaledSF36opt, 
    yTestSF36opt, 
    name="Logistic Regression",
    ax=ax[1],
    curve_kwargs={'color': 'forestgreen'}
)
ax[1].plot([0, 1], [0, 1], "k--", label="Chance Level (AUC = 0.5)")
ax[1].set_title("ROC Curve: Predicting Overuse (SF36)", fontsize=14)
ax[1].legend(loc='lower right') 

plt.tight_layout() 
plt.show()

print("\n--- Classification Report ---")
print(reportSF36opt)

balancedAccuracy, overallF1, brierScore, aucScore = computeAdditionalMetrics("SF36", yTestSF36opt, predictionSF36opt, yProbSF36opt)
print (f"Balanced Accuracy: {balancedAccuracy:.4f}")
print (f"Overall F1 Score: {overallF1:.4f}")
print (f"Brier Score: {brierScore:.4f}")
print (f"AUC Score: {aucScore:.4f}")

In [ ]:
plt.figure(figsize=(10, 8))
ax = plt.gca()

models = [
    {"model": logisticModelODI, "x": xTestScaledODI, "y": yTestODI, "label": "Logistic (ODI)", "color": "forestgreen"},
    {"model": logisticModelCOMI, "x": xTestScaledCOMI, "y": yTestCOMI, "label": "Logistic (COMI)", "color": "seagreen"},
    {"model": logisticModelSF36, "x": xTestScaledSF36, "y": yTestSF36, "label": "Logistic (SF36)", "color": "mediumseagreen"}
]


for m in models:
    RocCurveDisplay.from_estimator(
        m["model"], 
        m["x"], 
        m["y"], 
        name=m["label"],
        ax=ax, 
        color=m["color"]
    )

ax.plot([0, 1], [0, 1], "k--", label="Chance Level (AUC = 0.5)") 
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve Comparison', fontsize=15)
ax.legend(loc="lower right")
ax.grid(alpha=0.3)

plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
ax = plt.gca()

models = [
    {"model": logisticModelODI, "x": xTestScaledODI, "y": yTestODI, "label": "Logistic (ODI)", "color": "forestgreen"},
    {"model": logisticModelCOMI, "x": xTestScaledCOMI, "y": yTestCOMI, "label": "Logistic (COMI)", "color": "blue"},
    {"model": logisticModelSF36, "x": xTestScaledSF36, "y": yTestSF36, "label": "Logistic (SF36)", "color": "orange"}
]


for m in models:
    PrecisionRecallDisplay.from_estimator(
        m["model"], 
        m["x"], 
        m["y"], 
        name=m["label"],
        ax=ax, 
        color=m["color"]
    )

ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve Comparison', fontsize=15)
ax.legend(loc="lower right")
ax.grid(alpha=0.3)

plt.show()

SVM

In [ ]:
def svmModel(questionnaire: str):
    xTrainScaled, xTestScaled, yTrain, yTest = prepareData(questionnaire)

    model = SVC(kernel='linear', probability=True, random_state=883) 
    model.fit(xTrainScaled, yTrain)

    prediction = model.predict(xTestScaled)
    yProb = model.predict_proba(xTestScaled)[:, 1]

    return model, xTestScaled, yTest, prediction, yProb, confusion_matrix(yTest, prediction), classification_report(yTest, prediction)

In [ ]:
def svmModelKernel(questionnaire: str):
    xTrainScaled, xTestScaled, yTrain, yTest = prepareData(questionnaire)

    params = [
        {'kernel': ['linear'], 'C': [0.1, 1, 10, 100]},
        {'kernel': ['rbf'], 'C': [0.1, 1, 10, 100], 'gamma': ['scale', 'auto', 0.1, 0.01]},
        {'kernel': ['poly'], 'C': [0.1, 1, 10], 'degree': [2, 3], 'gamma': ['scale', 'auto']}
    ]

    baseModel = SVC(class_weight='balanced', probability=True, random_state=42)

    gridSearch = GridSearchCV(
        estimator=baseModel, 
        param_grid=params, 
        cv=5, 
        scoring='balanced_accuracy', 
        n_jobs=-1
    )

    gridSearch.fit(xTrainScaled, yTrain)

    print(f"Best params: {gridSearch.best_params_}\n")

    bestModel = gridSearch.best_estimator_

    prediction = bestModel.predict(xTestScaled)
    yProb = bestModel.predict_proba(xTestScaled)[:, 1]

    return bestModel, xTestScaled, yTest, prediction, yProb, confusion_matrix(yTest, prediction), classification_report(yTest, prediction)

In [ ]:
def evaluateSVM(questionaire: str, isWithKernel: bool):
    if isWithKernel:
        model, xTestScaled, yTest, prediction, yProb, confusionMatrix, report = svmModelKernel(questionaire)
        modelType = "SVM with kernel"
    else:
        model, xTestScaled, yTest, prediction, yProb, confusionMatrix, report = svmModel(questionaire)
        modelType = "Linear SVM"

    fig, ax = plt.subplots(1, 2, figsize=(16, 6))

    sns.heatmap(confusionMatrix, annot=True, fmt='d', cmap='Greens', cbar=False, ax=ax[0])
    ax[0].set_xlabel('Predicted Label')
    ax[0].set_ylabel('True Label')
    ax[0].set_title(f'Confusion Matrix ({modelType})')

    RocCurveDisplay.from_estimator(
        model, 
        xTestScaled, 
        yTest, 
        name=modelType,
        ax=ax[1],
        color='forestgreen'
    )
    ax[1].plot([0, 1], [0, 1], "k--", label="Chance Level (AUC = 0.5)")
    ax[1].set_title(f"ROC Curve: Predicting Overuse ({questionaire})", fontsize=14)
    ax[1].legend(loc='lower right') 

    plt.tight_layout() 
    plt.show()

    print("\n--- Classification Report ---")
    print(report)

    balancedAccuracy, overallF1, brierScore, aucScore = computeAdditionalMetrics(questionaire, yTest, prediction, yProb)
    print (f"Balanced Accuracy: {balancedAccuracy:.4f}")
    print (f"Overall F1 Score: {overallF1:.4f}")
    print (f"Brier Score: {brierScore:.4f}")
    print (f"AUC Score: {aucScore:.4f}")

In [ ]:
evaluateSVM("ODI", isWithKernel=False)

In [ ]:
evaluateSVM("ODI", isWithKernel=True)

In [ ]:
evaluateSVM("COMI", isWithKernel=False)

In [ ]:
evaluateSVM("COMI", isWithKernel=True)

In [ ]:
evaluateSVM("SF36", isWithKernel=False)

In [ ]:
evaluateSVM("SF36", isWithKernel=True)

DECISION TREES (XGBoost)

In [ ]:
def xgboostModel(questionnaire: str):
    xTrainScaled, xTestScaled, yTrain, yTest = prepareData(questionnaire)

    # weighting the positive class to handle imbalance (if pos_count is 0, weight is set to 1 to avoid division by zero, but in that case the model won't be able to learn anything about the positive class)
    neg_count = np.sum(yTrain == 0)
    pos_count = np.sum(yTrain == 1)
    weight = neg_count / pos_count if pos_count > 0 else 1

    params = {
        'learning_rate': [0.01, 0.1, 0.2],
        'max_depth': [3, 5],           
        'n_estimators': [100, 200],     
        'subsample': [0.6, 0.8, 1.0],        
        'colsample_bytree': [0.6, 0.8, 1.0]
    }

    baseXGB = XGBClassifier(
        scale_pos_weight=weight,
        eval_metric='logloss',
        random_state=42
    )

    gridSearch = GridSearchCV(
        estimator=baseXGB,
        param_grid=params,
        cv=5, 
        scoring='balanced_accuracy', 
        n_jobs=-1 
    )

    gridSearch.fit(xTrainScaled, yTrain)
    
    print(f"Best params: {gridSearch.best_params_}\n")

    bestModel = gridSearch.best_estimator_
    prediction = bestModel.predict(xTestScaled)
    yProb = bestModel.predict_proba(xTestScaled)[:, 1]
    
  
    return bestModel, xTestScaled, yTest, prediction, yProb, confusion_matrix(yTest, prediction), classification_report(yTest, prediction)

In [ ]:
def evaluateXGBoost(questionaire: str):
    model, xTestScaled, yTest, prediction, yProb, confusionMatrix, report = xgboostModel(questionaire)

    fig, ax = plt.subplots(1, 2, figsize=(16, 6))

    sns.heatmap(confusionMatrix, annot=True, fmt='d', cmap='Greens', cbar=False, ax=ax[0])
    ax[0].set_xlabel('Predicted Label')
    ax[0].set_ylabel('True Label')
    ax[0].set_title(f'Confusion Matrix - XGBoost')

    RocCurveDisplay.from_estimator(
        model, 
        xTestScaled, 
        yTest, 
        name="XGBoost",
        ax=ax[1],
        color='forestgreen'
    )
    ax[1].plot([0, 1], [0, 1], "k--", label="Chance Level (AUC = 0.5)")
    ax[1].set_title(f"ROC Curve: Predicting Overuse ({questionaire})", fontsize=14)
    ax[1].legend(loc='lower right') 

    plt.tight_layout() 
    plt.show()

    print("\n--- Classification Report ---")
    print(report)

    balancedAccuracy, overallF1, brierScore, aucScore = computeAdditionalMetrics(questionaire, yTest, prediction, yProb)
    print (f"Balanced Accuracy: {balancedAccuracy:.4f}")
    print (f"Overall F1 Score: {overallF1:.4f}")
    print (f"Brier Score: {brierScore:.4f}")
    print (f"AUC Score: {aucScore:.4f}")

In [ ]:
evaluateXGBoost("ODI")

In [ ]:
evaluateXGBoost("COMI")

In [ ]:
evaluateXGBoost("SF36")